# 🌌 NeoWatch — Aşama 1: Veri Boru Hattı (Data Pipeline) ve İstek Yönetimi

**Plan Belgesi**: NASA Proje Planı 2 (Aşama 1)

### 🎯 Hedefler:
1. **Güvenlik (Credentials Management)**: `.env` dosyasından `NASA_API_KEY` değişkenini güvenli şekilde yükleme (`python-dotenv`).
2. **Limit Yönetimi (Rate Limiting) & Döngüler**: NASA NeoWs API'nin 7 günlük sorgu kısıtına karşı ardışık sliding-window döngüsü tasarlama ve `time.sleep()` ile HTTP 429 engellemesini önleme.
3. **Özellik Ayıklama (Parsing) & Yapılandırma**: İç içe geçmiş JSON yapısını düzleştirerek (`id`, `name`, `estimated_diameter_min_km`, `estimated_diameter_max_km`, `relative_velocity_km_s`, `miss_distance_km`, `is_potentially_hazardous_asteroid`) tablosunu oluşturma.
4. **Checkpoint 1 (Veri Temini)**: Çekilen ham veriyi `data/raw_data.csv` dizinine kaydetme.

In [ ]:
import sys
import os
from pathlib import Path

# Proje ana dizinini sys.path'e ekleme
project_root = Path(os.path.abspath('')).parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import numpy as np
from src.config import NASA_API_KEY, RAW_DATA_PATH, LEGACY_RAW_DATA_PATH
from src.api_client import NASAClient

print("NASA API Key Durumu:", "Hazır (" + NASA_API_KEY[:6] + "...)" if NASA_API_KEY else "Bulunamadı!")

## 1. NASA NeoWs API Bağlantı Testi (7 Günlük Tekil İstek)

In [ ]:
client = NASAClient()

# 7 günlük test isteği
sample_json = client.fetch_feed_chunk(start_date="2023-01-01", end_date="2023-01-07")
print(f"API'den dönen cisim sayısı: {sample_json.get('element_count')}")

# JSON verisini düzleştirme (Parsing)
sample_records = client.parse_feed_json(sample_json)
sample_df = pd.DataFrame(sample_records)
sample_df.head(5)

## 2. Tarihler Arasında İterasyon (1 Yıllık Ham Veri Çekimi)

In [ ]:
# 1 yıllık zaman aralığında sliding-window döngüsü
start_date = "2023-01-01"
end_date = "2024-01-01"

print(f"Veri çekimi başlatılıyor: {start_date} -> {end_date}...")
df_raw = client.fetch_date_range(start_date, end_date, delay_between_calls=0.25)
print(f"Toplam çekilen ham veri boyutu: {df_raw.shape}")
df_raw.head()

## 3. CHECKPOINT 1: Veri Temini ve Kayıt

In [ ]:
# Veri tipleri ve eksik değer kontrolü
print("--- Veri Seti Özeti ---")
df_raw.info()

print("\n--- Hedef Değişken Dağılımı (is_potentially_hazardous_asteroid) ---")
target_counts = df_raw['is_potentially_hazardous_asteroid'].value_counts()
print(target_counts)
print(f"Tehlikeli Asteroit Oranı: %{df_raw['is_potentially_hazardous_asteroid'].mean()*100:.2f}")

# Checkpoint 1 Kaydı: data/raw_data.csv
client.save_to_csv(df_raw, RAW_DATA_PATH)
if RAW_DATA_PATH != LEGACY_RAW_DATA_PATH:
    client.save_to_csv(df_raw, LEGACY_RAW_DATA_PATH)

print(f"\n🎯 CHECKPOINT 1 TAMAMLANDI: Ham veri başarıyla kaydedildi -> {RAW_DATA_PATH}")